# Neural SDE — Option Pricing

---
## 0. Imports

In [119]:
import numpy as np
import pandas as pd
import yfinance as yf
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchsde
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

---
## 1. 데이터 로드 및 확인

In [78]:
df = pd.read_csv("data/nasdaq100_options_raw.csv")

features = ["S", "Strike", "Tau", "Sigma", "Call"]
target   = "Price"

df = df[features + [target]].dropna().copy()
df

,S,Strike,Tau,Sigma,Call,Price
0,293.320007,295.0,0.090411,0.232499,1,8.000
1,293.320007,290.0,0.090411,0.232499,1,10.725
2,293.320007,300.0,0.090411,0.232499,1,5.825
3,293.320007,295.0,0.090411,0.232499,0,8.975
4,293.320007,290.0,0.090411,0.232499,0,6.700
...,...,...,...,...,...,...
1775,152.130005,155.0,2.104110,0.471443,1,56.250
1776,152.130005,145.0,2.104110,0.471443,1,59.750
1777,152.130005,150.0,2.104110,0.471443,0,44.850
1778,152.130005,155.0,2.104110,0.471443,0,47.750


In [ ]:
def evaluate(y_true, y_pred, name):

    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)

    # MAPE (%)
    eps = 1e-8
    mape = np.mean(
        np.abs((y_true - y_pred) / (y_true + eps))
    ) * 100

    print(name)
    print(f"MAE  : {mae:.4f}")
    print(f"MSE  : {mse:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAPE : {mape:.2f}%")
    print(f"R2   : {r2:.4f}")
    print()

In [76]:
irx = yf.Ticker("^IRX")
hist = irx.history(
    start="2026-01-01",
    end="2026-04-01"
)
r_value = hist["Close"].mean() / 100
print(r_value)

0.03583098352932539


---
## 2. 공통 Train / Test Split

In [85]:
X = df[features].values.astype(np.float32)
y = df[target].values.astype(np.float32).reshape(-1, 1)

idx = np.arange(len(df))

# first split: train 80 / temp 20
X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(
    X, y, idx,
    test_size=0.2,
    random_state=42,
    shuffle=True
)


# second split: temp -> val 10 / test 10
X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(
    X_temp, y_temp, idx_temp,
    test_size=0.5,
    random_state=42,
    shuffle=True
)

print(X_train.shape, X_val.shape, X_test.shape)

(1424, 5) (178, 5) (178, 5)


---
## 3. Black-Scholes (BS)

In [87]:
def black_scholes_price(S, K, T, sigma, call, r=0.0):
    eps   = 1e-12
    S     = np.asarray(S, dtype=float)
    K     = np.asarray(K, dtype=float)
    T     = np.maximum(np.asarray(T, dtype=float), eps)
    sigma = np.maximum(np.asarray(sigma, dtype=float), eps)
    call  = np.asarray(call)

    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    call_price = S * norm.cdf(d1)  - K * np.exp(-r * T) * norm.cdf(d2)
    put_price  = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    return np.where(call == 1, call_price, put_price)

In [88]:
df_test = df.iloc[idx_test].copy()

bs_pred_test = black_scholes_price(
    S=df_test["S"],
    K=df_test["Strike"],
    T=df_test["Tau"],
    sigma=df_test["Sigma"],
    call=df_test["Call"],
    r=r_value
).reshape(-1, 1)

In [90]:
evaluate(y_test, bs_pred_test, "Pure BS")

Pure BS
MAE  : 9.3188
MSE  : 292.5440
RMSE : 17.1039
MAPE : 19.98%
R2   : 0.9532



---
## 4. ANN1 (Artificial Neural Network)

In [95]:
x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_s = x_scaler.fit_transform(X_train)

X_val_s = x_scaler.transform(X_val)
X_test_s = x_scaler.transform(X_test)

y_train_s = y_scaler.fit_transform(y_train)

y_val_s = y_scaler.transform(y_val)
y_test_s = y_scaler.transform(y_test)

In [96]:
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train_s, dtype=torch.float32)

X_val_t = torch.tensor(X_val_s, dtype=torch.float32)
y_val_t = torch.tensor(y_val_s, dtype=torch.float32)

X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_test_t = torch.tensor(y_test_s, dtype=torch.float32)

In [97]:
train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val_t, y_val_t),
    batch_size=16,
    shuffle=False
)

In [98]:
class ANN1(nn.Module):

    def __init__(self, input_dim=5, hidden_dim=32):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Sigmoid(),

            nn.Linear(hidden_dim, hidden_dim),
            nn.Sigmoid(),

            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.net(x)

In [99]:
torch.manual_seed(42)

model = ANN1(input_dim=5, hidden_dim=32)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)
loss_fn = nn.MSELoss()

n_epochs = 1000

best_val_loss = np.inf
best_state = None

for epoch in range(n_epochs):

    # train
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            pred = model(xb)
            loss = loss_fn(pred, yb)
            val_loss += loss.item()
    val_loss /= len(val_loader)

    # save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = model.state_dict()

    # print
    if epoch % 100 == 0:
        print(
            f"Epoch {epoch:4d} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f}"
        )

Epoch    0 | Train Loss: 0.914596 | Val Loss: 0.905039
Epoch  100 | Train Loss: 0.030591 | Val Loss: 0.018076
Epoch  200 | Train Loss: 0.025672 | Val Loss: 0.015558
Epoch  300 | Train Loss: 0.021352 | Val Loss: 0.013620
Epoch  400 | Train Loss: 0.021489 | Val Loss: 0.013000
Epoch  500 | Train Loss: 0.018727 | Val Loss: 0.012247
Epoch  600 | Train Loss: 0.017375 | Val Loss: 0.013891
Epoch  700 | Train Loss: 0.017360 | Val Loss: 0.014215
Epoch  800 | Train Loss: 0.016652 | Val Loss: 0.012900
Epoch  900 | Train Loss: 0.016651 | Val Loss: 0.015236


In [100]:
model.load_state_dict(best_state)
model.eval()

with torch.no_grad():
    ann1_pred_test_s = model(X_test_t).numpy()

# inverse transform
ann1_pred_test = y_scaler.inverse_transform(
    ann1_pred_test_s
)

In [101]:
evaluate(y_test, ann1_pred_test, "ANN1")

ANN1
MAE  : 6.8256
MSE  : 236.4489
RMSE : 15.3769
MAPE : 32.29%
R2   : 0.9622



---
## 5. Neural SDE

In [114]:
features = ["S", "Strike", "Tau", "Sigma", "Call"]
target = "Price"

X = df[features].values.astype(np.float32)
y = df[target].values.astype(np.float32).reshape(-1, 1)

idx = np.arange(len(df))

feature_idx = {name: i for i, name in enumerate(features)}
print(feature_idx)

{'S': 0, 'Strike': 1, 'Tau': 2, 'Sigma': 3, 'Call': 4}


In [115]:
# 3. TensorDataset maker
def make_sde_dataset(X, y, r_value):
    S = torch.tensor(X[:, feature_idx["S"]], dtype=torch.float32)
    K = torch.tensor(X[:, feature_idx["Strike"]], dtype=torch.float32)
    T = torch.tensor(X[:, feature_idx["Tau"]], dtype=torch.float32)
    call = torch.tensor(X[:, feature_idx["Call"]], dtype=torch.float32)

    if "Sigma" in feature_idx:
        hv = torch.tensor(X[:, feature_idx["Sigma"]], dtype=torch.float32)
    else:
        hv = torch.zeros_like(S)

    r = torch.full_like(S, r_value)
    price = torch.tensor(y.squeeze(), dtype=torch.float32)

    return TensorDataset(S, K, T, call, r, hv, price)

train_dataset = make_sde_dataset(X_train, y_train, r_value)
val_dataset   = make_sde_dataset(X_val, y_val, r_value)
test_dataset  = make_sde_dataset(X_test, y_test, r_value)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [116]:
# 4. Neural SDE model
class NeuralSDE(nn.Module):
    noise_type = "diagonal"
    sde_type = "ito"

    def __init__(self, hidden_dim=32, sigma_min=1e-4):
        super().__init__()

        self.sigma_min = sigma_min

        self.vol_net = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
            nn.Softplus()
        )

        self.K = None
        self.T = None
        self.r = None
        self.hist_vol = None

    def set_condition(self, K, T, r, hist_vol):
        self.K = K
        self.T = T
        self.r = r
        self.hist_vol = hist_vol

    def sigma_theta(self, u, x):
        S_t = torch.exp(torch.clamp(x, min=-20, max=20))

        log_moneyness = torch.log(
            torch.clamp(S_t / self.K, min=1e-8, max=1e8)
        )

        tau_remaining = (1.0 - u) * self.T

        phi = torch.cat(
            [log_moneyness, tau_remaining, self.hist_vol],
            dim=1
        )

        sigma = self.vol_net(phi) + self.sigma_min
        sigma = torch.clamp(sigma, min=1e-4, max=5.0)

        return sigma

    def f(self, u, x):
        sigma = self.sigma_theta(u, x)
        drift_x = self.T * (self.r - 0.5 * sigma ** 2)
        return drift_x

    def g(self, u, x):
        sigma = self.sigma_theta(u, x)
        diffusion_x = torch.sqrt(torch.clamp(self.T, min=1e-8)) * sigma
        return diffusion_x

In [117]:
# 5. Neural SDE pricing
def neural_sde_price_batch(
    sde,
    S0,
    K,
    T,
    call,
    r,
    hist_vol,
    n_paths=50,
    n_steps=10,
    use_adjoint=False,
    method="euler"
):
    device = next(sde.parameters()).device

    S0 = S0.to(device).float().view(-1, 1)
    K = K.to(device).float().view(-1, 1)
    T = T.to(device).float().view(-1, 1)
    call = call.to(device).float().view(-1, 1)
    r = r.to(device).float().view(-1, 1)
    hist_vol = hist_vol.to(device).float().view(-1, 1)

    batch_size = S0.shape[0]

    S0_rep = S0.repeat_interleave(n_paths, dim=0)
    K_rep = K.repeat_interleave(n_paths, dim=0)
    T_rep = T.repeat_interleave(n_paths, dim=0)
    r_rep = r.repeat_interleave(n_paths, dim=0)
    hist_vol_rep = hist_vol.repeat_interleave(n_paths, dim=0)

    X0_rep = torch.log(torch.clamp(S0_rep, min=1e-8))

    sde.set_condition(
        K=K_rep,
        T=T_rep,
        r=r_rep,
        hist_vol=hist_vol_rep
    )

    ts = torch.linspace(0.0, 1.0, n_steps, device=device)

    solver = torchsde.sdeint_adjoint if use_adjoint else torchsde.sdeint

    paths_x = solver(
        sde,
        X0_rep,
        ts,
        method=method
    )

    X_T = torch.clamp(paths_x[-1, :, 0], min=-20, max=20)
    S_T = torch.exp(X_T).view(batch_size, n_paths)

    K_mat = K.view(batch_size, 1)
    call_mat = call.view(batch_size, 1)

    call_payoff = torch.clamp(S_T - K_mat, min=0.0)
    put_payoff = torch.clamp(K_mat - S_T, min=0.0)

    payoff = call_mat * call_payoff + (1.0 - call_mat) * put_payoff

    discount = torch.exp(-r.view(batch_size, 1) * T.view(batch_size, 1))

    price = discount.squeeze(1) * payoff.mean(dim=1)
    price = torch.clamp(price, min=1e-6, max=1e6)

    return price

In [118]:
# 6. Log price loss only
def log_price_loss(pred_price, true_price, eps=1e-6):
    pred_price = torch.clamp(pred_price, min=eps, max=1e6)
    true_price = torch.clamp(true_price, min=eps, max=1e6)

    loss = torch.mean(
        (torch.log(pred_price) - torch.log(true_price)) ** 2
    )

    return loss

In [120]:
# 7. Train with validation
def train_sde_model_logloss(
    sde_model,
    optimizer,
    train_loader,
    val_loader,
    train_dataset,
    val_dataset,
    n_epochs=30,
    n_paths_train=50,
    n_steps_train=10,
    n_paths_val=100,
    n_steps_val=10,
    print_every=5
):
    train_loss_history = []
    val_loss_history = []

    best_val_loss = np.inf
    best_state = None

    for epoch in range(n_epochs):

        epoch_start = time.time()

        # =================================================
        # Train
        # =================================================
        sde_model.train()
        total_train_loss = 0.0

        for batch_idx, (
            S_batch,
            K_batch,
            T_batch,
            call_batch,
            r_batch,
            hv_batch,
            price_batch
        ) in enumerate(train_loader):

            batch_start = time.time()

            S_batch = S_batch.to(device)
            K_batch = K_batch.to(device)
            T_batch = T_batch.to(device)
            call_batch = call_batch.to(device)
            r_batch = r_batch.to(device)
            hv_batch = hv_batch.to(device)
            price_batch = price_batch.to(device)

            optimizer.zero_grad()

            pred_price = neural_sde_price_batch(
                sde=sde_model,
                S0=S_batch,
                K=K_batch,
                T=T_batch,
                call=call_batch,
                r=r_batch,
                hist_vol=hv_batch,
                n_paths=n_paths_train,
                n_steps=n_steps_train,
                use_adjoint=True,
                method="euler"
            )

            loss = log_price_loss(pred_price, price_batch)

            if not torch.isfinite(loss):
                print("NaN loss skipped")
                continue

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                sde_model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            total_train_loss += loss.item() * len(S_batch)

            # =================================================
            # Batch print
            # =================================================
            if (batch_idx + 1) % print_every == 0:
                batch_time = time.time() - batch_start

                print(
                    f"Epoch [{epoch+1:03d}/{n_epochs:03d}] | "
                    f"Batch [{batch_idx+1:03d}/{len(train_loader):03d}] | "
                    f"Loss: {loss.item():.6f} | "
                    f"Batch Time: {batch_time:.1f} sec"
                )

        avg_train_loss = total_train_loss / len(train_dataset)
        train_loss_history.append(avg_train_loss)

        # =================================================
        # Validation
        # =================================================
        sde_model.eval()
        total_val_loss = 0.0

        with torch.no_grad():

            for S_batch, K_batch, T_batch, call_batch, r_batch, hv_batch, price_batch in val_loader:

                pred_price = neural_sde_price_batch(
                    sde=sde_model,
                    S0=S_batch.to(device),
                    K=K_batch.to(device),
                    T=T_batch.to(device),
                    call=call_batch.to(device),
                    r=r_batch.to(device),
                    hist_vol=hv_batch.to(device),
                    n_paths=n_paths_val,
                    n_steps=n_steps_val,
                    use_adjoint=False,
                    method="euler"
                )

                price_batch = price_batch.to(device)

                val_loss = log_price_loss(pred_price, price_batch)

                total_val_loss += val_loss.item() * len(S_batch)

        avg_val_loss = total_val_loss / len(val_dataset)
        val_loss_history.append(avg_val_loss)

        # =================================================
        # Save best model by val loss
        # =================================================
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in sde_model.state_dict().items()
            }

        epoch_time = time.time() - epoch_start

        print(
            f"Epoch [{epoch+1:03d}/{n_epochs:03d}] Finished | "
            f"Train Loss: {avg_train_loss:.6f} | "
            f"Val Loss: {avg_val_loss:.6f} | "
            f"Time: {epoch_time:.1f} sec"
        )

    sde_model.load_state_dict(best_state)

    return train_loss_history, val_loss_history, best_val_loss

In [ ]:
# 8. Train model
torch.manual_seed(42)

sde_model = NeuralSDE(
    hidden_dim=32,
    sigma_min=1e-4
).to(device)

optimizer = optim.Adam(
    sde_model.parameters(),
    lr=5e-4
)

train_loss_history, val_loss_history, best_val_loss = train_sde_model_logloss(
    sde_model=sde_model,
    optimizer=optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    n_epochs=30,
    n_paths_train=50,
    n_steps_train=10,
    n_paths_val=100,
    n_steps_val=10
)

print("Best Val Loss:", best_val_loss)

Epoch [001/030] | Batch [005/089] | Loss: 0.446539 | Batch Time: 16.0 sec
Epoch [001/030] | Batch [010/089] | Loss: 0.563285 | Batch Time: 20.4 sec
Epoch [001/030] | Batch [015/089] | Loss: 0.758947 | Batch Time: 16.8 sec
Epoch [001/030] | Batch [020/089] | Loss: 0.493561 | Batch Time: 16.0 sec
Epoch [001/030] | Batch [025/089] | Loss: 0.851782 | Batch Time: 16.1 sec
Epoch [001/030] | Batch [030/089] | Loss: 0.330063 | Batch Time: 17.2 sec
Epoch [001/030] | Batch [035/089] | Loss: 0.504712 | Batch Time: 16.2 sec
Epoch [001/030] | Batch [040/089] | Loss: 0.313371 | Batch Time: 16.2 sec
Epoch [001/030] | Batch [045/089] | Loss: 0.149173 | Batch Time: 16.4 sec
Epoch [001/030] | Batch [050/089] | Loss: 0.109294 | Batch Time: 16.6 sec
Epoch [001/030] | Batch [055/089] | Loss: 0.259062 | Batch Time: 19.1 sec
Epoch [001/030] | Batch [060/089] | Loss: 0.209109 | Batch Time: 14.2 sec
Epoch [001/030] | Batch [065/089] | Loss: 0.136077 | Batch Time: 14.1 sec
Epoch [001/030] | Batch [070/089] | Lo

In [ ]:
# =====================================================
# 9. Save log-loss version separately
# =====================================================
torch.save(
    sde_model.state_dict(),
    "data/sde_model_logloss_811.pth"
)

torch.save(
    {
        "model_state_dict": sde_model.state_dict(),
        "hidden_dim": 32,
        "sigma_min": 1e-4,
        "lr": 5e-4,
        "split": "8:1:1",
        "loss": "log_price_loss",
        "n_epochs": 30,
        "n_paths_train": 100,
        "n_steps_train": 20,
        "n_paths_val": 200,
        "n_steps_val": 30,
        "best_val_loss": best_val_loss,
    },
    "data/sde_checkpoint_logloss_811.pth"
)

np.save(
    "data/sde_train_loss_history_logloss_811.npy",
    np.array(train_loss_history)
)

np.save(
    "data/sde_val_loss_history_logloss_811.npy",
    np.array(val_loss_history)
)

In [ ]:
# 10. Test evaluation
sde_model.eval()

sde_pred_list = []
sde_true_list = []

with torch.no_grad():

    for S_batch, K_batch, T_batch, call_batch, r_batch, hv_batch, price_batch in test_loader:

        pred_price = neural_sde_price_batch(
            sde=sde_model,
            S0=S_batch.to(device),
            K=K_batch.to(device),
            T=T_batch.to(device),
            call=call_batch.to(device),
            r=r_batch.to(device),
            hist_vol=hv_batch.to(device),
            n_paths=500,
            n_steps=40,
            use_adjoint=False,
            method="euler"
        )

        sde_pred_list.append(pred_price.cpu().numpy())
        sde_true_list.append(price_batch.numpy())

sde_pred_test = np.concatenate(sde_pred_list).reshape(-1, 1)
y_true_test_sde = np.concatenate(sde_true_list).reshape(-1, 1)

np.save("data/sde_pred_test_logloss_811.npy", sde_pred_test)
np.save("data/y_true_test_sde_811.npy", y_true_test_sde)

np.savez(
    "data/sde_results_test_logloss_811.npz",
    pred=sde_pred_test,
    true=y_true_test_sde,
    split="8:1:1",
    loss="log_price_loss",
    idx_test=idx_test
)

Neural SDE
MAE  : 5.4001
MSE  : 63.6666
RMSE : 7.9791
MAPE : 20.74%
R2   : 0.7926


---
## 6. 모델 비교

### 5-1. 지표 비교

In [56]:
evaluate(y_true_test, ann1_pred_test, "ANN1")
evaluate(y_true_test, bs_pred_test, "Pure BS")
evaluate(y_true_test, sde_pred_test, "Neural SDE")

ANN1
MAE  : 3.7795
MSE  : 27.5849
RMSE : 5.2521
MAPE : 16.78%
R2   : 0.9102

Pure BS
MAE  : 6.3433
MSE  : 99.5431
RMSE : 9.9771
MAPE : 24.20%
R2   : 0.6758

Neural SDE
MAE  : 4.4137
MSE  : 43.4922
RMSE : 6.5949
MAPE : 18.29%
R2   : 0.8584

